# rasterio tests

This notebook exemplifies how to calculate width and height of the final image to export, based on CRS, a bounding box and a scale (pixel resolution in meters, epsg:3857). In the end, a constant COG image is written.

In [36]:
import rasterio
import rasterio.windows
import rasterio.transform
import math
from pyproj import CRS, Transformer
from shapely.ops import transform
from shapely.geometry import box
from rio_cogeo.profiles import cog_profiles
import numpy as np

In [37]:
# Input parameters
crs = CRS.from_epsg(4326)
bbox = box(-1.8074931391317692, 49.6341724045420634, -1.8041198869927890,49.6353304980805987)
scale = 10  # in meters (epsg:3857)

In [38]:
dst_crs = CRS.from_epsg(3857)

project = Transformer.from_crs(crs, dst_crs, always_xy=True).transform
repr_box = transform(project, bbox)

repr_box.bounds

(-201209.21586048414, 6383160.344350464, -200833.7071500555, 6383359.397562613)

In [39]:
minx, miny, maxx, maxy = repr_box.bounds

affine = rasterio.transform.from_origin(west=minx, north=maxy, xsize=scale, ysize=scale)
window = rasterio.windows.from_bounds(left=minx, bottom=miny, right=maxx, top=maxy, transform=affine)

In [40]:
width, height = math.ceil(window.width), math.ceil(window.height)

In [41]:
width, height

(38, 20)

In [60]:
img = np.ones((width, height), dtype=np.uint8) * 255

In [62]:
west, south, east, north = bbox.bounds
affine = rasterio.transform.from_bounds(west, south, east, north, width, height)

profile = cog_profiles["deflate"].copy()
profile.update(dtype=img.dtype, count=1, crs=crs, transform=affine, width=img.shape[0], height=img.shape[1])
profile

{'driver': 'GTiff', 'interleave': 'pixel', 'tiled': True, 'blockxsize': 512, 'blockysize': 512, 'compress': 'DEFLATE', 'dtype': dtype('uint8'), 'count': 1, 'crs': <Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich
, 'transform': Affine(8.876979313105815e-05, 0.0, -1.8074931391317692,
       0.0, -5.7904676926767705e-05, 49.6353304980806), 'width': 38, 'height': 20}

In [63]:
dst_path = "constant.tif"
with rasterio.open(dst_path, "w", **profile) as dst:
    dst.write(img, 1)
    
print(dst_path, "written")

constant.tif written


In [64]:
!gdalinfo $dst_path

Driver: GTiff/GeoTIFF
Files: constant.tif
       constant.tif.aux.xml
Size is 38, 20
Coordinate System is:
GEOGCRS["WGS 84",
    DATUM["World Geodetic System 1984",
        ELLIPSOID["WGS 84",6378137,298.257223563,
            LENGTHUNIT["metre",1]]],
    PRIMEM["Greenwich",0,
        ANGLEUNIT["degree",0.0174532925199433]],
    CS[ellipsoidal,2],
        AXIS["geodetic latitude (Lat)",north,
            ORDER[1],
            ANGLEUNIT["degree",0.0174532925199433]],
        AXIS["geodetic longitude (Lon)",east,
            ORDER[2],
            ANGLEUNIT["degree",0.0174532925199433]],
    ID["EPSG",4326]]
Data axis to CRS axis mapping: 2,1
Origin = (-1.807493139131769,49.635330498080599)
Pixel Size = (0.000088769793131,-0.000057904676927)
Metadata:
  AREA_OR_POINT=Area
Image Structure Metadata:
  COMPRESSION=DEFLATE
  INTERLEAVE=BAND
Corner Coordinates:
Upper Left  (  -1.8074931,  49.6353305) (  1d48'26.98"W, 49d38' 7.19"N)
Lower Left  (  -1.8074931,  49.6341724) (  1d48'26.98"W, 49d38